In [0]:
import pyspark.sql.functions as F

In [0]:
bronze_table = dbutils.jobs.taskValues.get(taskKey="RawtoBronze",key="bronze_table",debugValue="onlineretail_ws.bronze.online_retail")
dbutils.widgets.text("silver_table", defaultValue="onlineretail_ws.silver.online_retail")
silver_table = dbutils.widgets.get("silver_table")
dbutils.jobs.taskValues.set("silver_table",silver_table)

In [0]:
source_df=spark.table(silver_table)

In [0]:
print("Bronze:", bronze_table)
print("Silver:", silver_table)

In [0]:
silver_df=spark.table(bronze_table)

In [0]:
silver_df=silver_df.drop('ingestion_time','source_file','source_path')\
    .withColumn("Customer_ID",F.col("Customer_ID").cast('int'))\
        .withColumn("Description",F.trim(F.col("Description")))\
            .withColumn("InvoiceDate",F.to_date(F.col("InvoiceDate")))\
                .withColumn("Start_Date",F.current_date()-10)\
                    .withColumn("End_Date",F.lit(None).cast('date'))\
                        .withColumn("is_active",F.lit('Y'))\
                            .filter(F.col('Customer_ID').isNotNull())\
                                .filter(F.col('Invoice').isNotNull())

In [0]:
currency_data = [
    ("India", "INR", "Indian Rupee"),
    ("United States", "USD", "US Dollar"),
    ("United Kingdom", "GBP", "British Pound"),
    ("Germany", "EUR", "Euro"),
    ("France", "EUR", "Euro"),
    ("Japan", "JPY", "Japanese Yen"),
    ("China", "CNY", "Chinese Yuan"),
    ("Australia", "AUD", "Australian Dollar"),
    ("Canada", "CAD", "Canadian Dollar"),
    ("Singapore", "SGD", "Singapore Dollar"),
    ("Switzerland", "CHF", "Swiss Franc"),
    ("UAE", "AED", "UAE Dirham"),
    ("Saudi Arabia", "SAR", "Saudi Riyal"),
    ("Brazil", "BRL", "Brazilian Real"),
    ("Mexico", "MXN", "Mexican Peso"),
    ("South Korea", "KRW", "South Korean Won"),
    ("Indonesia", "IDR", "Indonesian Rupiah"),
    ("Malaysia", "MYR", "Malaysian Ringgit"),
    ("Thailand", "THB", "Thai Baht"),
    ("South Africa", "ZAR", "South African Rand")
]

currency_df = spark.createDataFrame(
    currency_data,
    ["country", "currency_code", "currency_name"]
)

In [0]:
silver_df=silver_df.join(F.broadcast(currency_df), on='Country', how='left').drop('currency_name')

In [0]:
silver_df=silver_df.withColumn("Description",F.regexp_replace(F.col("Description"), '""', "|"))\
    .withColumn("Description",F.regexp_replace(F.col("Description"),'"',""))

In [0]:
dupe_key = ["Invoice","StockCode","Customer_ID","InvoiceDate","Description"]

In [0]:
silver_df=silver_df.withColumn('Is_Cancelled',F.when(F.substr(F.col('Invoice'),F.lit(1),F.lit(1))=='C','Y').otherwise('N'))

In [0]:
silver_df=silver_df.dropDuplicates(subset=[*dupe_key])

In [0]:
spark.sql(f"""
    MERGE INTO {silver_table} AS TRG
    USING {{source_df}} AS SRC
    ON TRG.Invoice = SRC.Invoice AND TRG.StockCode = SRC.StockCode AND TRG.CUSTomer_ID = SRC.Customer_ID 
    AND TRG.is_active = 'Y'
    WHEN MATCHED AND
        hash(concat_ws('|', TRG.Country, TRG.StockCode, TRG.Customer_ID, TRG.InvoiceDate, TRG.Description))
        != hash(concat_ws('|', SRC.Country, SRC.StockCode, SRC.Customer_ID, SRC.InvoiceDate, SRC.Description))
    THEN UPDATE SET
        TRG.is_active = 'N',
        TRG.End_Date = current_date()
    WHEN NOT MATCHED THEN
    INSERT *
""", source_df=source_df)

In [0]:
spark.sql(f"select * from {silver_table} where is_active='N'").show(5)